In [1]:
def reverse_one_generation(s, rules):
    img   = {rhs: lhs for lhs, rhs in rules.items()}   # image -> source symbol
    ruled = set(rules)                                 # symbols that always expand
    n = len(s)
    print("img:", img)
    print("ruled:", ruled)
    dp = [None]*(n+1); dp[0] = ""                       # dp[i] = a preimage of s[:i]
    print("dp start:", dp)
    print("---")
    for i in range(n):
        print(i, end=": ")
        if dp[i] is None:
            print("dp[i] is None")
            continue
        for rhs, lhs in img.items():                   # match each RHS at position i
            if s.startswith(rhs, i) and dp[i+len(rhs)] is None:
                print("found ", rhs, "; ", sep="",end="")
                dp[i+len(rhs)] = dp[i] + lhs
                print("dp[", i+len(rhs), "]: ", dp[i+len(rhs)],sep="")
        ch = s[i]                                       # pass-through for ruleless symbols
        if ch not in ruled and dp[i+1] is None:
            dp[i+1] = dp[i] + ch
            print("added ", ch, "; ", dp[i+1],sep="")
    return dp[n]                                        # None if no valid previous generation

In [15]:
reverse_one_generation("1111[11[1[0]0]1[0]0]11[1[0]0]1[0]0", {"1": "11", "0": "1[0]0"})

img: {'11': '1', '1[0]': '0'}
ruled: {'1', '0'}
dp start: ['', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
---
0: found 11; dp[2]: 1
1: dp[i] is None
2: found 11; dp[4]: 11
3: dp[i] is None
4: added [; 11[
5: found 11; dp[7]: 11[1
6: dp[i] is None
7: added [; 11[1[
8: found 1[0]; dp[12]: 11[1[0
9: dp[i] is None
10: dp[i] is None
11: dp[i] is None
12: 13: dp[i] is None
14: dp[i] is None
15: dp[i] is None
16: dp[i] is None
17: dp[i] is None
18: dp[i] is None
19: dp[i] is None
20: dp[i] is None
21: dp[i] is None
22: dp[i] is None
23: dp[i] is None
24: dp[i] is None
25: dp[i] is None
26: dp[i] is None
27: dp[i] is None
28: dp[i] is None
29: dp[i] is None
30: dp[i] is None
31: dp[i] is None
32: dp[i] is None
33: dp[i] is None
['', None, '1', None, '11', '11[', None, '11[1', '11[1[', None, None, None, '11[1[0', None, None, None, None,

In [ ]:
def reverse_one_generation_speculative(s, rules):
    img   = {rhs: lhs for lhs, rhs in rules.items()}   # image -> source symbol
    ruled = set(rules)                                 # symbols that always expand
    n = len(s)
    print("img:", img)
    print("ruled:", ruled)
    dp = [None]*(n+1); dp[0] = ""                       # dp[i] = a preimage of s[:i]
    print("dp start:", dp)
    print("---")
    for i in range(n):
        print(i, end=": ")
        #if dp[i] is None:
        #    print("dp[i] is None")
        #    continue
        for rhs, lhs in img.items():                   # match each RHS at position i
            if s.startswith(rhs, i) and dp[i+len(rhs)] is None:
                print("found ", rhs, "; ", sep="",end="")
                dp[i+len(rhs)] = dp[i] + lhs
                print("dp[", i+len(rhs), "]: ", dp[i+len(rhs)],sep="")
        ch = s[i]                                       # pass-through for ruleless symbols
        if ch not in ruled and dp[i+1] is None:
            dp[i+1] = dp[i] + ch
            print("added ", ch, "; ", dp[i+1],sep="")
    return dp[n]                                        # None if no valid previous generation

In [10]:
def reverse_one_generation_linear(s, sub1, sub2, a, b):
    # BROKEN, bad implementation
    # O(n)*O(replace), works only with D0L
    # We assume that the first reduction we take is the most optimal (11 -> a, 1111 is reduced like [11][11])
    def do_reverse(rhs1, rhs2, lhs1, lhs2):
        s1 = s.replace(rhs1, lhs1)
        return s1.replace(rhs2, lhs2)

    # null rule checks
    if len(a) == 0 and len(b) == 0:
        return s
    if len(a) == 0:
        return s.replace(sub2, b)
    elif len(b) == 0:
        return s.replace(sub1, a)
    
    # forward
    fwd = do_reverse(sub1, sub2, a, b)
    # backwards
    back = do_reverse(sub2, sub1, b, a)

    if len(fwd) < len(back):
        return fwd
    else:
        return back

In [4]:
import re

In [5]:
def merge_idx_first(rhs1, rhs2, idx1, idx2):
    # merge indices while prioritizing idx1, returns idx2
    i = 0; j = 0
    n1 = len(rhs1); n2 = len(rhs2)
    while i < len(idx1) and j < len(idx2):
        start1 = idx1[i]
        start2 = idx2[j]
        end1 = start1 + n1
        end2 = start2 + n2
        # check for intersection  (we need to call this twice to do )
        # [   ( ]   )  or  [  (   )  ]
        check_inter = lambda a_start, a_end, b_start, b_end: (b_start < a_end and b_end > a_start) or (a_start < b_end and a_end > b_start)
        
        if check_inter(start1, end1, start2, end2) or check_inter(start2, end2, start1, end1):  # cfg/base.h : check_intersect_ranges()
            idx2.pop(j)  # inefficient
            continue  # do not increment j
        
        # check which counter to move
        if i + 1 >= len(idx1):
            j += 1
        elif j + 1 >= len(idx2):
            i += 1
        else:
            # check which start will be closer (is this a correct metric???)
            # do we have to calculate distance here?
            next1 = idx1[i + 1]
            next2 = idx2[j + 1]
            if next1 < next2:
                i += 1
            else:
                j += 1
    return idx2

In [7]:
def reverse_one_generation_RE(s, sub1, sub2, a, b):
    # ideally O(n)*O(replace), but here it's much worse, works only with D0L
    # We assume that the first reduction we take is the most optimal (11 -> a, 1111 is reduced like [11][11])

    # null rule checks
    if len(a) == 0 and len(b) == 0:
        return s
    if len(a) == 0:
        return s.replace(sub2, b)
    elif len(b) == 0:
        return s.replace(sub1, a)
    
    idx1 = [m.start() for m in re.finditer(re.escape(sub1), s)]
    idx2 = [m.start() for m in re.finditer(re.escape(sub2), s)]
    # merge indices
    # we need a strategy for handling collisions
    # =============

    print(idx1, idx2)

    # simple strategy: pick the largest rule
    if sub1 > sub2:
        idx2 = merge_idx_first(sub1, sub2, idx1, idx2)
    else:
        idx1 = merge_idx_first(sub2, sub1, idx2, idx1)

    print(idx1, idx2)
    
    # substitute the result (inefficient)
    i = len(idx1) - 1; j = len(idx2) - 1
    n1 = len(sub1); n2 = len(sub2)
    replace = lambda idx, lhs, n: s[:idx] + lhs + s[idx + n:]
    
    while i >= 0 or j >= 0:
        if i >= 0 and j >= 0:
            # we can do this, since idx are not overlapping
            if idx1[i] > idx2[j]:  # pick which one is larger, so that we don't corrupt the indices
                print("lhs")
                s = replace(idx1[i], a, n1)
                i -= 1
            else:
                print("rhs")
                s = replace(idx2[j], b, n2)
                j -= 1
        elif i >= 0:
            print("e lhs")
            s = replace(idx1[i], a, n1)
            i -= 1
        else: # j > 0
            print("e rhs")
            s = replace(idx2[j], b, n2)
            j -= 1
    return s

In [11]:
reverse_one_generation_linear("1111[11[1[0]0]1[0]0]11[1[0]0]1[0]0", "11", "1[0]0", "1", "0")

'11[1[0]0]1[0]0'

In [9]:
reverse_one_generation_RE("1111[11[1[0]0]1[0]0]11[1[0]0]1[0]0", "11", "1[0]0", "1", "0")

[0, 2, 5, 20] [8, 14, 23, 29]
[0, 2, 5, 20] [8, 14, 23, 29]
rhs
rhs
lhs
rhs
rhs
e lhs
e lhs
e lhs


'11[1[0]0]1[0]0'

In [44]:
merge_idx("abcd", "ab", [2], [0, 2, 4, 6, 10])

[0, 6, 10]